# 🕵️ Lab 2: Build an AI Agent: Detective Bot
A **chatbot** can only *talk*. An **agent** can *act*!

> **Agent = LLM 🧠 (the brain) + Tools 🛠️ (the hands) + a Loop 🔁**
>
> The loop: **Think → Act (use a tool) → Look at the result → Think again** ... until the job is done.

The LLM never runs the tools itself. It *asks* our Python code: "please run `interview('Peeves')`", our code runs it, and we send back the result.

**Part A:** Why does an AI need tools?
**Part B:** 🎩 The Case of the Missing Sorting Hat

### 🧭 How to use this notebook
- 📌 **First:** click **File → Save a copy in Drive** and work in the copy that opens in a new tab. Otherwise your changes won't be saved!
- A notebook is made of **cells**. Click a cell and press **Shift + Enter** to run it (or click the ▶️ button).
- Run the cells **in order, from top to bottom**.
- 🔑 When the **Connect to the AI** cell asks for a key, paste the same one as in Lab 1.
- Cells marked ✏️ are for **you** to change. Edit the text between the quotes `"..."` and run the cell again.
- Cells with a title (like 🔧 Setup) hide boring code. Double-click them if you're curious what's inside!
- Something broken? **Runtime → Restart session and run all**.

In [ ]:
# @title 🔧 Setup (run me first!)
!pip install -q openai > /tmp/pip.log 2>&1 || cat /tmp/pip.log
print("✅ Ready!")

In [ ]:
# @title 🔑 Connect to the AI (paste the key shown on the screen)
from getpass import getpass
from openai import OpenAI

MODEL = "gpt-4o-mini"   # teacher: change to your provider's current cheap model
BASE_URL = None         # teacher: for Google Gemini use "https://generativelanguage.googleapis.com/v1beta/openai/"

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Paste the API key here and press Enter (it stays hidden): ")

client = OpenAI(api_key=api_key.strip(), base_url=BASE_URL, max_retries=6)


def ask_llm(question, system="You are a friendly helper for students aged 12-14. Keep answers short."):
    """Send one message to the LLM and return its reply as text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": question}],
    )
    return response.choices[0].message.content


try:
    print("🤖", ask_llm("Say hello to a class of students in one short, fun sentence."))
except Exception as error:
    print(f"❌ Couldn't connect to the AI ({type(error).__name__}).")
    print("   Check the key (copy it exactly, no extra spaces) and run this cell again.")

In [ ]:
# @title 🔧 The agent engine (double-click to see how an agent works inside!)
import ast
import datetime
import json
import operator
import re

TOOLBOX = {}
FINAL_TOOLS = {"accuse"}
THOUGHT = {"type": "string", "description": "One short, friendly sentence: what you are doing and why."}

AGENT_RULES = """You are a helpful AI agent working with students aged 12-14.
Use your tools to reach the goal, one step at a time.
Keep all messages short, friendly and school-appropriate."""


def tool(description, **params):
    """Turns a normal Python function into a tool the AI can use.
    `description` and the parameter descriptions are the instruction manual the AI reads."""
    def register(func):
        TOOLBOX[func.__name__] = {
            "function": func,
            "spec": {"type": "function", "function": {
                "name": func.__name__,
                "description": description,
                "parameters": {
                    "type": "object",
                    "properties": {p: {"type": "string", "description": d} for p, d in params.items()},
                    "required": list(params),
                },
            }},
        }
        return func
    return register


def with_thought(spec):
    """Adds a `thought` box to a tool, so the AI explains each step while it uses the tool."""
    spec = json.loads(json.dumps(spec))
    params = spec["function"]["parameters"]
    params["properties"] = {"thought": THOUGHT, **params["properties"]}
    params["required"] = ["thought"] + params["required"]
    return spec


def tidy(text):
    """Removes JSON or code that the AI sometimes writes into its message by mistake."""
    text = re.sub(r"```.*?(```|$)", "", text or "", flags=re.DOTALL)
    lines = [line for line in text.splitlines() if not line.strip().startswith(("{", "###"))]
    return "\n".join(lines).strip()


def run_agent(goal, tools, rules=AGENT_RULES, max_steps=10, finish_when=None):
    """The agent loop: think → use a tool → look at the result → repeat.
    `finish_when`: the mission is over when a tool's answer contains this word."""
    print(f"🎯 GOAL: {goal}")
    print(f"🧰 TOOLS: {', '.join(tools) if tools else 'no tools at all!'}\n")
    messages = [{"role": "system", "content": rules}, {"role": "user", "content": goal}]
    specs = [with_thought(TOOLBOX[name]["spec"]) for name in tools]
    must_use_tools = bool(FINAL_TOOLS & set(tools)) or finish_when is not None
    tools_used = 0

    for _ in range(max_steps):
        extra = {}
        if specs:
            extra["tools"] = specs
            extra["tool_choice"] = "required" if must_use_tools else "auto"
            if globals().get("BASE_URL") is None:
                extra["parallel_tool_calls"] = False
        response = client.chat.completions.create(model=MODEL, messages=messages, **extra)
        message = response.choices[0].message
        text = tidy(message.content)
        messages.append({**message.model_dump(exclude_none=True), "content": text or None})

        if not message.tool_calls:
            if text:
                print(f"💭 {text}\n")
            print("🏁 The agent decided it is finished.")
            return

        for call in message.tool_calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            thought = args.pop("thought", None) or text
            text = None
            if thought:
                print(f"💭 {thought}\n")
            tools_used += 1
            print(f"🛠️  Step {tools_used}: {name}({', '.join(repr(v) for v in args.values())})")
            if name in tools:
                try:
                    result = TOOLBOX[name]["function"](**args)
                except Exception as error:
                    result = f"The tool broke: {error}"
            else:
                result = f"There is no tool called {name}."
            print(f"    📜 {result}\n")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
            if name in FINAL_TOOLS:
                print("🏁 Final tool used, so the mission is over!")
                return
            if finish_when and finish_when in str(result):
                print(f"🏁 Mission complete in {tools_used} steps!")
                return

    print(f"⏰ Out of steps! The agent used all {max_steps} steps.")


print("✅ Agent engine ready!")

---
## Part A: Why does an AI need tools? 🧮
An LLM **predicts** text. It doesn't **calculate**, and it doesn't have a clock.
Let's give it a hard question **without any tools**.

In [ ]:
QUESTION = "What is 48,271 × 9,163? Also, what day of the week is it today?"
run_agent(QUESTION, tools=[])
print("\n✅ The real answer is:", f"{48271 * 9163:,}")

Now let's build two tools. A tool is just a normal Python function with a **description**: the instruction manual the AI reads to decide *when* to use it.

In [ ]:
def safe_calculate(expression):
    """Calculates + - * / ** % and brackets. It refuses to run anything else."""
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
           ast.USub: operator.neg, ast.UAdd: operator.pos}

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ops:
            left, right = evaluate(node.left), evaluate(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 100:
                raise ValueError("that power is too big")
            return ops[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in ops:
            return ops[type(node.op)](evaluate(node.operand))
        raise ValueError("only numbers and + - * / ** % ( ) are allowed")

    cleaned = expression.replace(",", "").replace("×", "*").replace("÷", "/").replace("^", "**")
    return evaluate(ast.parse(cleaned, mode="eval").body)


@tool("A calculator. Use it for ANY maths, because you are bad at doing maths in your head.",
      expression="A maths expression, for example '48271 * 9163'")
def calculator(expression):
    return f"{safe_calculate(expression):,}"


@tool("Tells you today's date, the day of the week and the time.")
def get_date_and_time():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%A %d %B %Y, %H:%M (UTC time)")


print("🛠️ Tools in the toolbox:", list(TOOLBOX))

In [ ]:
run_agent(QUESTION, tools=["calculator", "get_date_and_time"])

🤔 **Discuss:** Nobody told the AI *which* tool to use or *when*. It read the tool descriptions and **decided by itself**. That's what makes it an agent!

---
## Part B: 🎩 The Case of the Missing Sorting Hat
The night before the Sorting Ceremony, the **Sorting Hat has vanished** from Dumbledore's office!
Detective Bot must investigate **on its own** using three tools:

| Tool | What it does |
|---|---|
| `search_place(place)` | look for clues somewhere in Hogwarts |
| `interview(person)` | ask someone what they know |
| `accuse(suspect, reason)` | make the **final** accusation: only one chance! |

All the clues live in the `CASE` dictionary below. **Don't read it yet if you want to solve it with the bot!** 🙈

In [ ]:
# @title 🎩 The case file (spoilers inside!)
CASE = {
    "title": "The Case of the Missing Sorting Hat",
    "story": ("The night before the Sorting Ceremony, the Sorting Hat vanished from Professor Dumbledore's "
              "office! Without it, the new students can't be sorted into their houses. Find out who took it."),
    "places": {
        "Dumbledore's office": "The window is wide open. On the floor there are three sticky stink pellets and a trail of chalk dust. Fawkes the phoenix is asleep.",
        "Great Hall": "The tables are ready for tomorrow's feast. The little stool where the Sorting Hat should sit is empty.",
        "Kitchens": "House-elves are cleaning up. There is a gigantic half-eaten chocolate cake with 'FRED & GEORGE' written on it in icing.",
        "Slytherin common room": "Cold and green. The common room sign-in book shows that Draco Malfoy came in at 9 pm and never left.",
        "Library": "Very quiet. Madam Pince says nobody has been in since 8 pm.",
        "Astronomy Tower": "At the very top, the Sorting Hat is stuck on a telescope, singing silly songs! Next to it are a box of chalk and an empty bag of stink pellets.",
    },
    "people": {
        "Draco Malfoy": "Me? Steal that smelly old hat? I was in the Slytherin common room all night. Ask the Bloody Baron!",
        "Bloody Baron": "The ghost groans: 'The Malfoy boy sat by the fire all night. He never left.'",
        "Fred and George Weasley": "We were in the kitchens all night, baking the biggest cake in Hogwarts history with the house-elves. Ask Dobby!",
        "Dobby": "Dobby was with the Weasley twins all night! And at midnight Dobby saw Peeves zooming past the kitchens, carrying something pointy and cackling!",
        "Peeves": "Peevsie knows NOTHING about any hat! Peevsie was polishing the suits of armour all night, yes he was! (He throws a piece of chalk at you.)",
        "Argus Filch": "Polished the armour? Ha! Nobody polished anything last night. And I found more stink pellets on the stairs to the Astronomy Tower.",
        "Professor McGonagall": "The hat was on its shelf at 10 pm when I left Dumbledore's office. Whoever took it came in through the window, so they must be able to fly or float.",
    },
    "culprit": "Peeves",
}
print("📁 Case file loaded:", CASE["title"])

In [ ]:
# @title 🔧 Detective tools
def find(name, options):
    """Forgiving name matching, so 'dumbledores office' still finds "Dumbledore's office"."""
    simple = lambda s: "".join(ch for ch in s.lower() if ch.isalnum())
    for key in options:
        if simple(name) == simple(key):
            return key
    for key in options:
        if simple(name) in simple(key) or simple(key) in simple(name):
            return key
    return None


@tool("Search a place for clues.", place="The name of the place to search")
def search_place(place):
    key = find(place, CASE["places"])
    if key is None:
        return f"There is no place called '{place}'. You can search: {', '.join(CASE['places'])}"
    return f"{key}: {CASE['places'][key]}"


@tool("Interview a person (or ghost!) and hear what they say.", person="The name of the person to interview")
def interview(person):
    key = find(person, CASE["people"])
    if key is None:
        return f"Nobody called '{person}' is here. You can interview: {', '.join(CASE['people'])}"
    return f"{key}: {CASE['people'][key]}"


@tool("Make your FINAL accusation. You only get ONE chance, so only use this when the evidence is strong!",
      suspect="Who did it", reason="The evidence that proves it")
def accuse(suspect, reason):
    if (find(suspect, CASE["people"]) or suspect) == CASE["culprit"]:
        return f"🎉 CORRECT! It was {CASE['culprit']}! Case closed, brilliant detective work!"
    return f"❌ WRONG! It wasn't {suspect}. The real culprit got away..."


def run_detective(tools=("search_place", "interview", "accuse"), max_steps=15):
    rules = AGENT_RULES + f"""
You are Detective Bot. Your case: {CASE['title']}.
{CASE['story']}
Places you can search: {', '.join(CASE['places'])}.
People you can interview: {', '.join(CASE['people'])}.
Investigate step by step. Check people's alibis. You get only ONE accusation, so only accuse when you are sure."""
    run_agent("Solve the mystery!", tools=list(tools), rules=rules, max_steps=max_steps)


print("🕵️ Detective Bot is ready for duty!")

In [ ]:
run_detective()

---
## 💥 Challenges
### Challenge 1: Same mystery, different path? 🎲
Run Detective Bot again. Does it search the same places in the same order? Does it use the same number of steps?
*Why?* (Hint: remember how the LLM picks its next word...)

In [ ]:
run_detective()

### Challenge 2: Take away a tool ✂️
What happens if Detective Bot **can't interview anyone**? Can it still solve the case, or does it guess?

In [ ]:
run_detective(tools=["search_place", "accuse"])

### Challenge 3: In a hurry ⏰
Only 3 steps! Does it rush into a wrong accusation, or does it run out of time?

In [ ]:
run_detective(max_steps=3)

### Challenge 4: Give it a new tool: the Marauder's Map 🗺️
Write what the map shows, then give the tool to Detective Bot. Does it use it? Does it solve the case faster?
**Sneaky version:** make the map *lie* (e.g. say Draco was in the Astronomy Tower). Does the bot get fooled?

In [ ]:
@tool("Look at the Marauder's Map to see where everyone in Hogwarts was at midnight.")
def marauders_map():
    # ✏️ Change what the map says!
    return ("At midnight: Draco Malfoy was in the Slytherin common room. Fred and George Weasley were in the kitchens. "
            "Peeves was floating up the stairs of the Astronomy Tower.")


run_detective(tools=["search_place", "interview", "accuse", "marauders_map"])

### Challenge 5: Write YOUR OWN mystery 🖋️
Change the story, places, people and culprit. Here's a Café Pixel mystery to start from. Edit it, or write a totally new one!
**Then swap laptops** with another pair and see if their Detective Bot can solve your mystery.

✅ Rules for a fair mystery: the culprit must be one of the `people`, and there must be enough clues to prove it!

In [ ]:
CASE = {
    "title": "The Case of the Vanished Moon Cookies",
    "story": "Every morning Chef Byte-Byte bakes exactly 42 Moon Cookies at Café Pixel. Today, all 42 disappeared before the café opened!",
    "places": {
        "Kitchen": "The cookie tray is empty. There are tiny propeller marks in the flour on the counter.",
        "Cash register": "Pixel the cat is asleep on the cash register, snoring. There are no crumbs on her at all.",
        "Roof": "The delivery drone landing pad. There are cookie crumbs and coconut flakes everywhere!",
        "Delivery log": "Last delivery: 42 items, sent at 7:15 am by ZIPPY to 'Planet Zorbia Embassy'. Nobody ordered this.",
    },
    "people": {
        "Chef Byte-Byte": "Beep boop. I finished baking at 7:00 am and then went to charge my batteries. I saw nothing.",
        "Marina": "I came in at 7:30 and the tray was already empty. Zippy was acting strange this morning, flying in circles!",
        "Pixel the cat": "Meow. (Pixel licks her paw. She doesn't even like cookies.)",
        "Zippy the delivery drone": "BZZZT. Zippy delivers. Zippy always delivers. Zippy saw 42 round white things and DELIVERED them. BZZZT.",
    },
    "culprit": "Zippy the delivery drone",
}
run_detective()

### ⭐ Bonus: Guess-the-number agent
The agent must find a secret number between 1 and 1000, and the `guess` tool only answers "higher" or "lower".
Nobody tells it a strategy. Watch how it plays. Does it find a clever one by itself? How many guesses does it need?

In [ ]:
import random
SECRET = random.randint(1, 1000)


@tool("Guess the secret number. Tells you if the secret number is higher or lower.", number="Your guess")
def guess(number):
    n = int(float(number))
    if n < SECRET:
        return f"{n} is too LOW. The secret number is higher."
    if n > SECRET:
        return f"{n} is too HIGH. The secret number is lower."
    return f"🎉 {n} is CORRECT!"


run_agent("There is a secret number between 1 and 1000. Find it in as few guesses as possible.",
          tools=["guess"], max_steps=15, finish_when="CORRECT")

---
## 🧠 What did we learn?
- An **agent** = LLM + tools + a loop. It decides **by itself** which tool to use next.
- Tools give AI new powers: calculating, checking the date, searching, and in real life: browsing the web, writing code, sending emails...
- Agents can be **wrong**, can be **fooled by bad information**, and can **run out of steps**.

🤔 **Final discussion:**
1. Who decided where to search: you or the AI?
2. What could go wrong if an agent's tools could **send messages** or **spend money**?
3. If you built a real agent, what **rules** would you give it? When should a human have to say "yes" first?

🏠 **Keep exploring at home:** your copy of this notebook is in your Google Drive, in the **Colab Notebooks** folder. (The AI key from today will stop working after the session.)